# PlaceMux – Task 1: Student ↔ Job Matching
**AI/ML Engineer | Week 2 | Phase 2**

This notebook covers:
1. Load & explore data
2. Define the feature space
3. Build a baseline model
4. Build a logistic regression model
5. Evaluate with real metrics
6. Explain one match (plain English)
7. Define the matching API contract

## Cell 1 — Import libraries

In [31]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')
print('All libraries loaded successfully!')

All libraries loaded successfully!


## Cell 2 — Load the datasets

In [32]:
students = pd.read_csv('../datasets/students.csv')
jobs     = pd.read_csv('../datasets/jobs.csv')
matches  = pd.read_csv('../datasets/matches.csv')

print('Students:', students.shape)
print('Jobs    :', jobs.shape)
print('Matches :', matches.shape)
print()
print(students.head(3))
print()
print(jobs.head(3))
print()
print(matches.head(3))

Students: (20, 7)
Jobs    : (9, 7)
Matches : (180, 6)

   student_id                                skills  internship_months  \
0           1   Python:85,SQL:75,Excel:70,Pandas:80                 18   
1           2       Java:80,Spring:75,SQL:65,Git:70                 24   
2           3  Python:90,ML:85,TensorFlow:75,SQL:70                 12   

  education_level certifications     preferred_role   location  
0           BTech     Python,SQL       Data Analyst       Pune  
1              BE           Java  Backend Developer     Mumbai  
2             MCA             ML        ML Engineer  Bangalore  

   job_id company_name          job_title                required_skills  \
0     101     TechNova       Data Analyst      Python:70,SQL:60,Excel:50   
1     102    CodeWorks  Backend Developer       Java:70,Spring:65,SQL:60   
2     103      AI Labs        ML Engineer  Python:80,ML:70,TensorFlow:60   

   min_experience_years job_type   location  
0                     1   Hybrid    

## Cell 3 — Helper: parse skill strings

Both `students.skills` and `jobs.required_skills` are stored as `Python:85,SQL:75` strings.
This helper converts them into a dict like `{'Python': 85, 'SQL': 75}`.

In [33]:
def parse_skills(skill_str):
    """Convert 'Python:85,SQL:75' -> {'Python': 85, 'SQL': 75}"""
    result = {}
    if pd.isna(skill_str) or str(skill_str).strip() == '':
        return result
    for part in str(skill_str).split(','):
        part = part.strip()
        if ':' in part:
            skill, score = part.split(':', 1)
            try:
                result[skill.strip()] = int(score.strip())
            except ValueError:
                pass
    return result

# Test it
print(parse_skills('Python:85,SQL:75,Excel:70'))
print(parse_skills('Python:70,SQL:60'))

{'Python': 85, 'SQL': 75, 'Excel': 70}
{'Python': 70, 'SQL': 60}


## Cell 4 — Define the Feature Space

This is the core of the task. We compute three features for every student-job pair:

| Feature | What it means |
|---|---|
| `skill_overlap_ratio` | % of job's required skills the student actually meets (at or above the required score) |
| `experience_gap` | student's internship months / 6 minus job's min years (positive = student has more) |
| `education_level_encoded` | BE/BTech/MCA etc. turned into a number |

In [34]:
def compute_skill_overlap(student_skills_str, job_skills_str):
    """
    Returns:
      overlap_count  - how many required skills the student meets (at or above required score)
      overlap_ratio  - overlap_count / total required skills  (0.0 to 1.0)
    """
    student_skills = parse_skills(student_skills_str)
    required_skills = parse_skills(job_skills_str)

    if not required_skills:
        return 0, 0.0

    met = 0
    for skill, required_score in required_skills.items():
        student_score = student_skills.get(skill, 0)
        if student_score >= required_score:
            met += 1

    return met, met / len(required_skills)


def compute_experience_gap(internship_months, min_exp_years):
    """
    Convert internship months to years (rough: 6 months = 1 year)
    Positive gap = student has more experience than needed
    """
    student_exp_years = internship_months / 6.0
    return round(student_exp_years - min_exp_years, 2)


# Quick sanity test
count, ratio = compute_skill_overlap('Python:85,SQL:75,Excel:70,Pandas:80', 'Python:70,SQL:60,Excel:50')
print(f'Overlap count: {count}, Overlap ratio: {ratio:.2f}')

gap = compute_experience_gap(12, 1)
print(f'Experience gap: {gap}')

Overlap count: 3, Overlap ratio: 1.00
Experience gap: 1.0


## Cell 5 — Build the full feature matrix from matches.csv

We join students and jobs into the matches table and compute features for each row.
If your matches.csv already has `skill_overlap_ratio` and `experience_gap`, we recalculate
them from scratch so numbers are consistent.

In [35]:
# Merge matches with student and job info
df = matches.merge(students, on='student_id', how='left')
df = df.merge(jobs, on='job_id', how='left')

print('Merged shape:', df.shape)
print('Columns:', df.columns.tolist())
print()

# Recalculate features cleanly
overlap_results = df.apply(
    lambda row: compute_skill_overlap(row['skills'], row['required_skills']), axis=1
)
df['skill_overlap_count_calc'] = [r[0] for r in overlap_results]
df['skill_overlap_ratio_calc'] = [r[1] for r in overlap_results]

df['experience_gap_calc'] = df.apply(
    lambda row: compute_experience_gap(row['internship_months'], row['min_experience_years']), axis=1
)

# Encode education level
le = LabelEncoder()
df['education_encoded'] = le.fit_transform(df['education_level'].fillna('Unknown'))
print('Education classes:', list(le.classes_))

print(df[['student_id','job_id','skill_overlap_ratio_calc','experience_gap_calc','education_encoded','label']].head(5))

Merged shape: (180, 18)
Columns: ['student_id', 'job_id', 'skill_overlap_count', 'skill_overlap_ratio', 'experience_gap', 'label', 'skills', 'internship_months', 'education_level', 'certifications', 'preferred_role', 'location_x', 'company_name', 'job_title', 'required_skills', 'min_experience_years', 'job_type', 'location_y']

Education classes: ['BE', 'BTech', 'MCA']
   student_id  job_id  skill_overlap_ratio_calc  experience_gap_calc  \
0           1     101                  1.000000                  2.0   
1           1     102                  0.333333                  1.0   
2           1     103                  0.333333                  2.0   
3           1     104                  0.666667                  2.0   
4           1     105                  0.000000                  2.0   

   education_encoded  label  
0                  1      1  
1                  1      0  
2                  1      0  
3                  1      1  
4                  1      0  


## Cell 6 — Train / Test Split

In [36]:
FEATURES = ['skill_overlap_ratio_calc', 'experience_gap_calc', 'education_encoded']
TARGET   = 'label'

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size : {len(X_train)}')
print(f'Test size  : {len(X_test)}')
print(f'Label distribution in train:\n{y_train.value_counts()}')
print(f'Label distribution in test:\n{y_test.value_counts()}')

Train size : 144
Test size  : 36
Label distribution in train:
label
0    126
1     18
Name: count, dtype: int64
Label distribution in test:
label
0    32
1     4
Name: count, dtype: int64


## Cell 7 — Baseline Model

**Dumb baseline**: rank purely by `skill_overlap_ratio`. Predict match (1) if ratio >= 0.5, else no match (0).
Every later model must beat this number.

In [37]:
BASELINE_THRESHOLD = 0.5

y_pred_baseline = (X_test['skill_overlap_ratio_calc'] >= BASELINE_THRESHOLD).astype(int)

print('=== BASELINE RESULTS (skill overlap ratio >= 0.5) ===')
print(f"Precision : {precision_score(y_test, y_pred_baseline):.3f}")
print(f"Recall    : {recall_score(y_test, y_pred_baseline):.3f}")
print()

cm = confusion_matrix(y_test, y_pred_baseline)
tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, cm[0][0])
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
print(f"False Positive Rate : {fpr:.3f}")
print(f"Confusion Matrix:\n{cm}")

=== BASELINE RESULTS (skill overlap ratio >= 0.5) ===
Precision : 1.000
Recall    : 1.000

False Positive Rate : 0.000
Confusion Matrix:
[[32  0]
 [ 0  4]]


## Cell 8 — Logistic Regression Model (Smarter)

Uses all three features. Should beat the baseline on precision/recall.

In [38]:
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train, y_train)

y_pred_lr = model.predict(X_test)

print('=== LOGISTIC REGRESSION RESULTS ===')
print(f"Precision : {precision_score(y_test, y_pred_lr):.3f}")
print(f"Recall    : {recall_score(y_test, y_pred_lr):.3f}")
print()

cm_lr = confusion_matrix(y_test, y_pred_lr)
tn, fp, fn, tp = cm_lr.ravel() if cm_lr.size == 4 else (0, 0, 0, cm_lr[0][0])
fpr_lr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
print(f"False Positive Rate : {fpr_lr:.3f}")
print(f"Confusion Matrix:\n{cm_lr}")
print()
print('Full report:')
print(classification_report(y_test, y_pred_lr))

=== LOGISTIC REGRESSION RESULTS ===
Precision : 0.000
Recall    : 0.000

False Positive Rate : 0.000
Confusion Matrix:
[[32  0]
 [ 4  0]]

Full report:
              precision    recall  f1-score   support

           0       0.89      1.00      0.94        32
           1       0.00      0.00      0.00         4

    accuracy                           0.89        36
   macro avg       0.44      0.50      0.47        36
weighted avg       0.79      0.89      0.84        36



## Cell 9 — Feature Importance (Explainability)

Which feature matters most? Logistic regression coefficients tell us directly.

In [39]:
coef_df = pd.DataFrame({
    'Feature'    : FEATURES,
    'Coefficient': model.coef_[0]
}).sort_values('Coefficient', ascending=False)

print('=== FEATURE IMPORTANCE (Logistic Regression Coefficients) ===')
print(coef_df.to_string(index=False))
print()
print('A higher positive coefficient = stronger signal for a good match.')

=== FEATURE IMPORTANCE (Logistic Regression Coefficients) ===
                 Feature  Coefficient
skill_overlap_ratio_calc     4.405602
       education_encoded     0.075538
     experience_gap_calc    -0.075713

A higher positive coefficient = stronger signal for a good match.


## Cell 10 — Plain-English Explanation for One Match

This is your demo moment. Pick one student + one job, show the match score, explain WHY.

In [40]:
def explain_match(student_id, job_id):
    """
    Give a plain-English explanation for why a student does or doesn't match a job.
    """
    student = students[students['student_id'] == student_id].iloc[0]
    job     = jobs[jobs['job_id'] == job_id].iloc[0]

    s_skills  = parse_skills(student['skills'])
    j_skills  = parse_skills(job['required_skills'])
    count, ratio = compute_skill_overlap(student['skills'], job['required_skills'])
    exp_gap   = compute_experience_gap(student['internship_months'], job['min_experience_years'])
    edu_enc   = le.transform([student['education_level']])[0]

    # Predict
    features  = np.array([[ratio, exp_gap, edu_enc]])
    pred      = model.predict(features)[0]
    prob      = model.predict_proba(features)[0][1]

    # Print explanation
    print('=' * 55)
    print(f'STUDENT  : {student_id}')
    print(f'JOB      : {job_id} — {job["job_title"]} at {job["company_name"]}')
    print('=' * 55)
    print(f"Student's skills  : {student['skills']}")
    print(f"Required skills   : {job['required_skills']}")
    print()

    print('--- Skill-by-skill breakdown ---')
    for skill, req_score in j_skills.items():
        student_score = s_skills.get(skill, 0)
        met = ' MET' if student_score >= req_score else ' NOT MET'
        print(f'  {skill:<12} required {req_score:>3}   student has {student_score:>3}   {met}')

    print()
    print(f'Skill overlap     : {count}/{len(j_skills)} skills met  ({ratio*100:.1f}%)')
    print(f'Experience gap    : {exp_gap:+.1f} years  (positive = student has more than required)')
    print(f'Education         : {student["education_level"]} (encoded: {edu_enc})')
    print()
    print(f'MATCH SCORE (probability) : {prob:.3f}')
    print(f'PREDICTION                : {" GOOD MATCH" if pred == 1 else " NOT A MATCH"}')
    print()
    print('Plain-English reason:')
    if pred == 1:
        print(f'  This student meets {count} out of {len(j_skills)} required skills at the needed score level.')
        if exp_gap >= 0:
            print(f'  Their internship experience is sufficient for this role.')
        print(f'  The model gives this pair a {prob*100:.1f}% match confidence.')
    else:
        missing = [s for s, r in j_skills.items() if s_skills.get(s, 0) < r]
        print(f'  The student is missing or underscore on: {", ".join(missing)}.')
        if exp_gap < 0:
            print(f'  Experience is also {abs(exp_gap):.1f} years short of what the job needs.')
        print(f'  Match confidence is only {prob*100:.1f}%.')


# Change these IDs to any real ones from your CSV
DEMO_STUDENT_ID = students['student_id'].iloc[0]
DEMO_JOB_ID     = jobs['job_id'].iloc[0]

explain_match(DEMO_STUDENT_ID, DEMO_JOB_ID)

STUDENT  : 1
JOB      : 101 — Data Analyst at TechNova
Student's skills  : Python:85,SQL:75,Excel:70,Pandas:80
Required skills   : Python:70,SQL:60,Excel:50

--- Skill-by-skill breakdown ---
  Python       required  70   student has  85    MET
  SQL          required  60   student has  75    MET
  Excel        required  50   student has  70    MET

Skill overlap     : 3/3 skills met  (100.0%)
Experience gap    : +2.0 years  (positive = student has more than required)
Education         : BTech (encoded: 1)

MATCH SCORE (probability) : 0.740
PREDICTION                :  GOOD MATCH

Plain-English reason:
  This student meets 3 out of 3 required skills at the needed score level.
  Their internship experience is sufficient for this role.
  The model gives this pair a 74.0% match confidence.


## Cell 11 — Matching API Contract

This is the spec you hand to the Backend team.
It defines exactly what goes IN and what comes OUT of your matching function.

In [41]:
import json

api_contract = {
    "endpoint"    : "POST /api/v1/match",
    "description" : "Returns a match score and plain-English reason for a student-job pair.",
    "request_body": {
        "student_id"        : "string  — e.g. 'S001'",
        "skills"            : "string  — e.g. 'Python:85,SQL:75,Excel:70'",
        "internship_months" : "int     — e.g. 12",
        "education_level"   : "string  — e.g. 'BE'",
        "job_id"            : "string  — e.g. 'J001'",
        "required_skills"   : "string  — e.g. 'Python:70,SQL:60'",
        "min_experience_years" : "float — e.g. 1.0"
    },
    "response_body": {
        "student_id"         : "string",
        "job_id"             : "string",
        "match_score"        : "float (0.0 to 1.0) — model's match probability",
        "prediction"         : "int   — 1 = good match, 0 = no match",
        "skill_overlap_ratio": "float — fraction of required skills met",
        "experience_gap"     : "float — positive = student has more experience than needed",
        "reason"             : "string — plain-English explanation"
    },
    "example_request": {
        "student_id"           : "S001",
        "skills"               : "Python:85,SQL:75,Excel:70,Pandas:80",
        "internship_months"    : 12,
        "education_level"      : "BE",
        "job_id"               : "J001",
        "required_skills"      : "Python:70,SQL:60,Excel:50",
        "min_experience_years" : 1.0
    },
    "example_response": {
        "student_id"          : "S001",
        "job_id"              : "J001",
        "match_score"         : 0.87,
        "prediction"          : 1,
        "skill_overlap_ratio" : 1.0,
        "experience_gap"      : 1.0,
        "reason"              : "Student meets all 3 required skills at or above the required score. Experience is sufficient. Match confidence: 87%."
    },
    "error_cases": {
        "400" : "Missing required fields",
        "404" : "student_id or job_id not found",
        "500" : "Model inference failed"
    }
}

print('=== MATCHING API CONTRACT ===')
print(json.dumps(api_contract, indent=2))

=== MATCHING API CONTRACT ===
{
  "endpoint": "POST /api/v1/match",
  "description": "Returns a match score and plain-English reason for a student-job pair.",
  "request_body": {
    "student_id": "string  \u2014 e.g. 'S001'",
    "skills": "string  \u2014 e.g. 'Python:85,SQL:75,Excel:70'",
    "internship_months": "int     \u2014 e.g. 12",
    "education_level": "string  \u2014 e.g. 'BE'",
    "job_id": "string  \u2014 e.g. 'J001'",
    "required_skills": "string  \u2014 e.g. 'Python:70,SQL:60'",
    "min_experience_years": "float \u2014 e.g. 1.0"
  },
  "response_body": {
    "student_id": "string",
    "job_id": "string",
    "match_score": "float (0.0 to 1.0) \u2014 model's match probability",
    "prediction": "int   \u2014 1 = good match, 0 = no match",
    "skill_overlap_ratio": "float \u2014 fraction of required skills met",
    "experience_gap": "float \u2014 positive = student has more experience than needed",
    "reason": "string \u2014 plain-English explanation"
  },
  "ex

## Cell 12 — Summary: Baseline vs Model comparison

In [42]:
summary = pd.DataFrame({
    'Model'    : ['Baseline (overlap >= 0.5)', 'Logistic Regression'],
    'Precision': [
        round(precision_score(y_test, y_pred_baseline), 3),
        round(precision_score(y_test, y_pred_lr), 3)
    ],
    'Recall'   : [
        round(recall_score(y_test, y_pred_baseline), 3),
        round(recall_score(y_test, y_pred_lr), 3)
    ],
    'FPR'      : [
        round(fpr, 3),
        round(fpr_lr, 3)
    ]
})

print('=== FINAL COMPARISON ===')
print(summary.to_string(index=False))
print()
print('Precision = of the matches we predicted, how many were actually good?')
print('Recall    = of all the actual good matches, how many did we catch?')
print('FPR       = false alarm rate — how often we said match when it wasn\'t')

=== FINAL COMPARISON ===
                    Model  Precision  Recall  FPR
Baseline (overlap >= 0.5)        1.0     1.0  0.0
      Logistic Regression        0.0     0.0  0.0

Precision = of the matches we predicted, how many were actually good?
Recall    = of all the actual good matches, how many did we catch?
FPR       = false alarm rate — how often we said match when it wasn't


# Task 2 — Job Posting with Skill Thresholds
**Focus:** Build the matching feature vectors from real job thresholds.

This continues from Task 1 using the same `students.csv` / `jobs.csv`. No new dataset needed —
we convert the existing skill strings into fixed-length numeric vectors and validate L1-L100 thresholds.

## Cell 13 — Build the Master Skill List

Scan every skill across students.csv and jobs.csv to get one fixed, ordered list.
Every vector (student or job) will follow this exact order.

In [43]:
all_skills = set()

for skill_str in students['skills'].dropna():
    all_skills.update(parse_skills(skill_str).keys())

for skill_str in jobs['required_skills'].dropna():
    all_skills.update(parse_skills(skill_str).keys())

MASTER_SKILLS = sorted(all_skills)
print(f'Total unique skills found: {len(MASTER_SKILLS)}')
print(MASTER_SKILLS)

Total unique skills found: 35
['.NET', 'API', 'AWS', 'Android', 'C#', 'C++', 'CSS', 'Cybersecurity', 'DSA', 'Data Science', 'Django', 'Docker', 'Excel', 'Flask', 'Git', 'HTML', 'Java', 'JavaScript', 'Kotlin', 'Linux', 'ML', 'Microservices', 'MongoDB', 'Networking', 'Node', 'Pandas', 'PowerBI', 'Python', 'REST', 'React', 'SQL', 'Spring', 'Statistics', 'Tableau', 'TensorFlow']


## Cell 14 — Convert Skills to Vectors

Each student/job becomes a fixed-length array. Missing skill = 0.
Example: if MASTER_SKILLS = ['Excel','Pandas','Python','SQL'],
then 'Python:85,SQL:75' -> [0, 0, 85, 75]

In [44]:
def skills_to_vector(skill_str, master_list=MASTER_SKILLS):
    """Convert a skill string into a fixed-length numeric vector."""
    skill_dict = parse_skills(skill_str)
    return np.array([skill_dict.get(skill, 0) for skill in master_list])


# Test
test_vec = skills_to_vector('Python:85,SQL:75')
print('Master skill list :', MASTER_SKILLS)
print('Vector             :', test_vec)

Master skill list : ['.NET', 'API', 'AWS', 'Android', 'C#', 'C++', 'CSS', 'Cybersecurity', 'DSA', 'Data Science', 'Django', 'Docker', 'Excel', 'Flask', 'Git', 'HTML', 'Java', 'JavaScript', 'Kotlin', 'Linux', 'ML', 'Microservices', 'MongoDB', 'Networking', 'Node', 'Pandas', 'PowerBI', 'Python', 'REST', 'React', 'SQL', 'Spring', 'Statistics', 'Tableau', 'TensorFlow']
Vector             : [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0 85  0  0 75  0  0  0  0]


## Cell 15 — Build Vectors for All Students and Jobs

In [45]:
students['skill_vector'] = students['skills'].apply(skills_to_vector)
jobs['threshold_vector']  = jobs['required_skills'].apply(skills_to_vector)

print('Sample student vector:')
print(students[['student_id', 'skills', 'skill_vector']].head(3))
print()
print('Sample job threshold vector:')
print(jobs[['job_id', 'required_skills', 'threshold_vector']].head(3))

Sample student vector:
   student_id                                skills  \
0           1   Python:85,SQL:75,Excel:70,Pandas:80   
1           2       Java:80,Spring:75,SQL:65,Git:70   
2           3  Python:90,ML:85,TensorFlow:75,SQL:70   

                                        skill_vector  
0  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 70, 0, 0,...  
1  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 70,...  
2  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  

Sample job threshold vector:
   job_id                required_skills  \
0     101      Python:70,SQL:60,Excel:50   
1     102       Java:70,Spring:65,SQL:60   
2     103  Python:80,ML:70,TensorFlow:60   

                                    threshold_vector  
0  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 50, 0, 0,...  
1  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  
2  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  


## Cell 16 — Cosine Similarity Between Student and Job Vectors

This is the same technique you used in your Skill Recommendation Engine project.
Cosine similarity gives a score from 0 (no match) to 1 (perfect direction match).

In [46]:
def cosine_similarity_manual(vec1, vec2):
    """Cosine similarity without needing required skills to be non-zero everywhere."""
    dot = np.dot(vec1, vec2)
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return dot / (norm1 * norm2)


def get_match_vector_score(student_id, job_id):
    s_vec = students[students['student_id'] == student_id]['skill_vector'].iloc[0]
    j_vec = jobs[jobs['job_id'] == job_id]['threshold_vector'].iloc[0]
    return cosine_similarity_manual(s_vec, j_vec), s_vec, j_vec


# Test on first student/job
sid = students['student_id'].iloc[0]
jid = jobs['job_id'].iloc[0]
score, s_vec, j_vec = get_match_vector_score(sid, jid)
print(f'Student {sid} vs Job {jid}')
print(f'Student vector : {s_vec}')
print(f'Job vector     : {j_vec}')
print(f'Cosine similarity: {score:.3f}')

Student 1 vs Job 101
Student vector : [ 0  0  0  0  0  0  0  0  0  0  0  0 70  0  0  0  0  0  0  0  0  0  0  0
  0 80  0 85  0  0 75  0  0  0  0]
Job vector     : [ 0  0  0  0  0  0  0  0  0  0  0  0 50  0  0  0  0  0  0  0  0  0  0  0
  0  0  0 70  0  0 60  0  0  0  0]
Cosine similarity: 0.856


## Cell 17 — Threshold Validation (L1-L100 mapping)

For every required skill, check if the student's score meets or exceeds the job's threshold.
We treat the raw 0-100 score directly as the L1-L100 level (L70 = score of 70).
Returns a pass/fail per skill plus an overall verdict.

In [47]:
def validate_thresholds(student_id, job_id):
    """
    Check each required skill threshold individually.
    Returns: dict with per-skill pass/fail, plus overall pass/fail.
    """
    student = students[students['student_id'] == student_id].iloc[0]
    job     = jobs[jobs['job_id'] == job_id].iloc[0]

    student_skills  = parse_skills(student['skills'])
    required_skills = parse_skills(job['required_skills'])

    results = {}
    for skill, required_level in required_skills.items():
        student_level = student_skills.get(skill, 0)
        results[skill] = {
            'required_level' : required_level,
            'student_level'  : student_level,
            'passed'         : student_level >= required_level
        }

    overall_passed = all(r['passed'] for r in results.values()) if results else False
    return results, overall_passed


# Test
results, overall = validate_thresholds(sid, jid)
for skill, r in results.items():
    status = 'PASS' if r['passed'] else 'FAIL'
    print(f"{skill:<12} required L{r['required_level']:<4} student L{r['student_level']:<4} [{status}]")
print()
print('OVERALL THRESHOLD RESULT:', 'PASS ' if overall else 'FAIL ')

Python       required L70   student L85   [PASS]
SQL          required L60   student L75   [PASS]
Excel        required L50   student L70   [PASS]

OVERALL THRESHOLD RESULT: PASS 


## Cell 18 — Run Vectors + Threshold Validation on ALL Student-Job Pairs

This builds the real-data evidence: every pair from matches.csv gets a vector similarity
score AND a threshold pass/fail, so we can measure precision/recall against the true label.

In [48]:
vector_results = []

for _, row in matches.iterrows():
    sid_i, jid_i, true_label = row['student_id'], row['job_id'], row['label']
    sim_score, _, _ = get_match_vector_score(sid_i, jid_i)
    _, threshold_passed = validate_thresholds(sid_i, jid_i)

    vector_results.append({
        'student_id'        : sid_i,
        'job_id'            : jid_i,
        'cosine_similarity' : round(sim_score, 3),
        'threshold_passed'  : threshold_passed,
        'true_label'        : true_label
    })

vector_df = pd.DataFrame(vector_results)
print(vector_df.head(10))

   student_id  job_id  cosine_similarity  threshold_passed  true_label
0         1.0   101.0              0.856              True         1.0
1         1.0   102.0              0.257             False         0.0
2         1.0   103.0              0.358             False         0.0
3         1.0   104.0              0.523             False         1.0
4         1.0   105.0              0.000             False         0.0
5         1.0   106.0              0.000             False         0.0
6         1.0   107.0              0.000             False         0.0
7         1.0   108.0              0.000             False         0.0
8         1.0   109.0              0.250             False         0.0
9         2.0   101.0              0.256             False         0.0


## Cell 19 — Evaluate Threshold Validation Against True Labels

Same metrics as Task 1: precision, recall, false-positive rate — but now using
threshold_passed as the prediction instead of the logistic regression model.

In [49]:
y_true       = vector_df['true_label']
y_pred_thresh = vector_df['threshold_passed'].astype(int)

print('=== THRESHOLD VALIDATION RESULTS (vs true labels) ===')
print(f"Precision : {precision_score(y_true, y_pred_thresh, zero_division=0):.3f}")
print(f"Recall    : {recall_score(y_true, y_pred_thresh, zero_division=0):.3f}")

cm_thresh = confusion_matrix(y_true, y_pred_thresh)
print(f'Confusion Matrix:\n{cm_thresh}')

if cm_thresh.size == 4:
    tn, fp, fn, tp = cm_thresh.ravel()
    fpr_thresh = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    print(f'False Positive Rate : {fpr_thresh:.3f}')

=== THRESHOLD VALIDATION RESULTS (vs true labels) ===
Precision : 1.000
Recall    : 0.455
Confusion Matrix:
[[158   0]
 [ 12  10]]
False Positive Rate : 0.000


## Cell 20 — Plain-English Demo: One Full Match Vector Walkthrough

Pick one student-job pair and show everything: vectors, similarity score, threshold check, verdict.

In [50]:
def demo_match_vector(student_id, job_id):
    sim_score, s_vec, j_vec = get_match_vector_score(student_id, job_id)
    results, overall_passed = validate_thresholds(student_id, job_id)

    print('=' * 55)
    print(f'MATCH VECTOR DEMO: {student_id} vs {job_id}')
    print('=' * 55)
    print(f'Skill order      : {MASTER_SKILLS}')
    print(f'Student vector   : {s_vec.tolist()}')
    print(f'Job threshold vec: {j_vec.tolist()}')
    print(f'Cosine similarity: {sim_score:.3f}')
    print()
    print('--- Threshold check (per skill) ---')
    for skill, r in results.items():
        status = 'PASS' if r['passed'] else 'FAIL'
        print(f"  {skill:<12} required L{r['required_level']:<4} student L{r['student_level']:<4} [{status}]")
    print()
    print(f'OVERALL: {"PASS  — eligible for this job" if overall_passed else "FAIL  — does not meet all thresholds"}')
    print(f'Vector similarity confidence: {sim_score*100:.1f}%')


# Run the demo on any pair you want to showcase
demo_match_vector(sid, jid)

MATCH VECTOR DEMO: 1 vs 101
Skill order      : ['.NET', 'API', 'AWS', 'Android', 'C#', 'C++', 'CSS', 'Cybersecurity', 'DSA', 'Data Science', 'Django', 'Docker', 'Excel', 'Flask', 'Git', 'HTML', 'Java', 'JavaScript', 'Kotlin', 'Linux', 'ML', 'Microservices', 'MongoDB', 'Networking', 'Node', 'Pandas', 'PowerBI', 'Python', 'REST', 'React', 'SQL', 'Spring', 'Statistics', 'Tableau', 'TensorFlow']
Student vector   : [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 70, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 80, 0, 85, 0, 0, 75, 0, 0, 0, 0]
Job threshold vec: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 50, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 70, 0, 0, 60, 0, 0, 0, 0]
Cosine similarity: 0.856

--- Threshold check (per skill) ---
  Python       required L70   student L85   [PASS]
  SQL          required L60   student L75   [PASS]
  Excel        required L50   student L70   [PASS]

OVERALL: PASS  — eligible for this job
Vector similarity confidence: 85.6%


## Cell 21 — Threshold→Competency Mapping (Validation)

Maps raw L1-L100 scores to competency bands so the founder/non-technical team
can understand what a number actually means.

In [51]:
def level_to_competency(level):
    """Map a raw L1-L100 score to a human-readable competency band."""
    if level >= 80:
        return 'Expert (L80-L100)'
    elif level >= 60:
        return 'Advanced (L60-L79)'
    elif level >= 40:
        return 'Intermediate (L40-L59)'
    elif level >= 20:
        return 'Beginner (L20-L39)'
    elif level > 0:
        return 'Novice (L1-L19)'
    else:
        return 'No exposure (L0)'


# Validate the mapping is consistent across all students' skills
mapping_check = []
for _, row in students.iterrows():
    skill_dict = parse_skills(row['skills'])
    for skill, level in skill_dict.items():
        mapping_check.append({
            'student_id' : row['student_id'],
            'skill'      : skill,
            'level'      : level,
            'competency' : level_to_competency(level)
        })

mapping_df = pd.DataFrame(mapping_check)
print(mapping_df.head(10))
print()
print('Competency band distribution:')
print(mapping_df['competency'].value_counts())

   student_id   skill  level          competency
0           1  Python     85   Expert (L80-L100)
1           1     SQL     75  Advanced (L60-L79)
2           1   Excel     70  Advanced (L60-L79)
3           1  Pandas     80   Expert (L80-L100)
4           2    Java     80   Expert (L80-L100)
5           2  Spring     75  Advanced (L60-L79)
6           2     SQL     65  Advanced (L60-L79)
7           2     Git     70  Advanced (L60-L79)
8           3  Python     90   Expert (L80-L100)
9           3      ML     85   Expert (L80-L100)

Competency band distribution:
competency
Expert (L80-L100)     35
Advanced (L60-L79)    30
Name: count, dtype: int64


## Cell 22 — Summary for Hand-off (Vectorization → Matching Service)

This is what you hand off to the matching service / backend team.

In [52]:
handoff_summary = {
    'master_skill_list'        : MASTER_SKILLS,
    'vector_length'             : len(MASTER_SKILLS),
    'vector_method'             : 'fixed-order numeric array, missing skill = 0',
    'similarity_method'        : 'cosine similarity',
    'threshold_rule'            : 'student_level >= required_level, per skill, ALL must pass',
    'competency_bands'         : ['Novice (1-19)', 'Beginner (20-39)', 'Intermediate (40-59)', 'Advanced (60-79)', 'Expert (80-100)'],
    'precision_on_sample_data' : round(precision_score(y_true, y_pred_thresh, zero_division=0), 3),
    'recall_on_sample_data'    : round(recall_score(y_true, y_pred_thresh, zero_division=0), 3)
}

print('=== HAND-OFF SUMMARY ===')
for k, v in handoff_summary.items():
    print(f'{k}: {v}')

=== HAND-OFF SUMMARY ===
master_skill_list: ['.NET', 'API', 'AWS', 'Android', 'C#', 'C++', 'CSS', 'Cybersecurity', 'DSA', 'Data Science', 'Django', 'Docker', 'Excel', 'Flask', 'Git', 'HTML', 'Java', 'JavaScript', 'Kotlin', 'Linux', 'ML', 'Microservices', 'MongoDB', 'Networking', 'Node', 'Pandas', 'PowerBI', 'Python', 'REST', 'React', 'SQL', 'Spring', 'Statistics', 'Tableau', 'TensorFlow']
vector_length: 35
vector_method: fixed-order numeric array, missing skill = 0
similarity_method: cosine similarity
threshold_rule: student_level >= required_level, per skill, ALL must pass
competency_bands: ['Novice (1-19)', 'Beginner (20-39)', 'Intermediate (40-59)', 'Advanced (60-79)', 'Expert (80-100)']
precision_on_sample_data: 1.0
recall_on_sample_data: 0.455


# Task 3 — Search & Discovery
**Focus:** Rank jobs for students and candidates for companies (v1).

Reuses everything from Task 1 & 2 — `students`, `jobs`, `MASTER_SKILLS`, `skills_to_vector()`,
`cosine_similarity_manual()`, `parse_skills()`, `validate_thresholds()`. No new data needed.

**Run this notebook top to bottom in one session** — Task 3 cells depend on functions defined in Task 1 & 2 cells above.

## Cell 23 — Baseline Ranking Function

Dumb baseline: rank purely by `skill_overlap_ratio` (no cosine similarity, no vectors).
Every smarter ranking must beat this.

In [53]:
def baseline_rank_jobs_for_student(student_id, top_n=None):
    """Rank all jobs for a student using simple skill overlap ratio (baseline)."""
    student = students[students['student_id'] == student_id].iloc[0]
    results = []
    for _, job in jobs.iterrows():
        count, ratio = compute_skill_overlap(student['skills'], job['required_skills'])
        results.append({
            'job_id': job['job_id'],
            'job_title': job['job_title'],
            'company_name': job['company_name'],
            'skill_overlap_ratio': ratio
        })
    ranked = pd.DataFrame(results).sort_values('skill_overlap_ratio', ascending=False).reset_index(drop=True)
    ranked.index += 1  # rank starts at 1
    return ranked.head(top_n) if top_n else ranked


def baseline_rank_students_for_job(job_id, top_n=None):
    """Rank all students for a job using simple skill overlap ratio (baseline)."""
    job = jobs[jobs['job_id'] == job_id].iloc[0]
    results = []
    for _, student in students.iterrows():
        count, ratio = compute_skill_overlap(student['skills'], job['required_skills'])
        results.append({
            'student_id': student['student_id'],
            'preferred_role': student['preferred_role'],
            'skill_overlap_ratio': ratio
        })
    ranked = pd.DataFrame(results).sort_values('skill_overlap_ratio', ascending=False).reset_index(drop=True)
    ranked.index += 1
    return ranked.head(top_n) if top_n else ranked


# Test baseline
print('=== BASELINE: Jobs ranked for Student 1 ===')
print(baseline_rank_jobs_for_student(1))

=== BASELINE: Jobs ranked for Student 1 ===
   job_id           job_title company_name  skill_overlap_ratio
1     101        Data Analyst     TechNova             1.000000
2     104          BI Analyst   DataVision             0.666667
3     102   Backend Developer    CodeWorks             0.333333
4     103         ML Engineer      AI Labs             0.333333
5     109   Software Engineer     SoftCore             0.333333
6     105  Frontend Developer     WebCraft             0.000000
7     106      Cloud Engineer  CloudSphere             0.000000
8     107    Security Analyst    SecureNet             0.000000
9     108   Android Developer  MobileWorks             0.000000


## Cell 24 — Smarter Ranking: Cosine Similarity on Match Vectors

Uses the same vectors from Task 2, ranks by cosine similarity instead of raw overlap ratio.
This accounts for *how much* a skill is met/missed, not just a binary pass/fail.

In [54]:
def rank_jobs_for_student(student_id, top_n=None):
    """Rank all jobs for a student using cosine similarity on skill vectors (smarter model)."""
    s_vec = students[students['student_id'] == student_id]['skill_vector'].iloc[0]
    results = []
    for _, job in jobs.iterrows():
        j_vec = job['threshold_vector']
        sim = cosine_similarity_manual(s_vec, j_vec)
        _, overall_passed = validate_thresholds(student_id, job['job_id'])
        results.append({
            'job_id': job['job_id'],
            'job_title': job['job_title'],
            'company_name': job['company_name'],
            'match_score': round(sim, 3),
            'threshold_passed': overall_passed
        })
    ranked = pd.DataFrame(results).sort_values('match_score', ascending=False).reset_index(drop=True)
    ranked.index += 1
    return ranked.head(top_n) if top_n else ranked


def rank_students_for_job(job_id, top_n=None):
    """Rank all students for a job using cosine similarity on skill vectors (smarter model)."""
    j_vec = jobs[jobs['job_id'] == job_id]['threshold_vector'].iloc[0]
    results = []
    for _, student in students.iterrows():
        s_vec = student['skill_vector']
        sim = cosine_similarity_manual(s_vec, j_vec)
        _, overall_passed = validate_thresholds(student['student_id'], job_id)
        results.append({
            'student_id': student['student_id'],
            'preferred_role': student['preferred_role'],
            'match_score': round(sim, 3),
            'threshold_passed': overall_passed
        })
    ranked = pd.DataFrame(results).sort_values('match_score', ascending=False).reset_index(drop=True)
    ranked.index += 1
    return ranked.head(top_n) if top_n else ranked


print('=== SMARTER: Jobs ranked for Student 1 (cosine similarity) ===')
print(rank_jobs_for_student(1))
print()
print('=== SMARTER: Candidates ranked for Job 101 (cosine similarity) ===')
print(rank_students_for_job(101))

=== SMARTER: Jobs ranked for Student 1 (cosine similarity) ===
   job_id           job_title company_name  match_score  threshold_passed
1     101        Data Analyst     TechNova        0.856              True
2     104          BI Analyst   DataVision        0.523             False
3     103         ML Engineer      AI Labs        0.358             False
4     102   Backend Developer    CodeWorks        0.257             False
5     109   Software Engineer     SoftCore        0.250             False
6     105  Frontend Developer     WebCraft        0.000             False
7     106      Cloud Engineer  CloudSphere        0.000             False
8     107    Security Analyst    SecureNet        0.000             False
9     108   Android Developer  MobileWorks        0.000             False

=== SMARTER: Candidates ranked for Job 101 (cosine similarity) ===
    student_id        preferred_role  match_score  threshold_passed
1            6          Data Analyst        0.985            

## Cell 25 — Evaluate Ranking Quality: Precision@K and Recall@K

Standard ranking metrics. For each student, check if their TRUE good-match jobs
(label=1 in matches.csv) actually appear in the top-K ranked results.

- **Precision@K** = of the top K jobs shown, how many are true good matches?
- **Recall@K** = of all true good matches for this student, how many appear in top K?

In [55]:
def precision_recall_at_k(k=3):
    """
    Computes Precision@K and Recall@K for job ranking, averaged across all students
    who have at least one true positive label in matches.csv.
    """
    precisions, recalls = [], []

    for student_id in students['student_id'].unique():
        true_positive_jobs = set(
            matches[(matches['student_id'] == student_id) & (matches['label'] == 1)]['job_id']
        )
        if not true_positive_jobs:
            continue  # skip students with no known good match

        ranked = rank_jobs_for_student(student_id, top_n=k)
        top_k_jobs = set(ranked['job_id'])

        hits = len(top_k_jobs & true_positive_jobs)
        precisions.append(hits / k)
        recalls.append(hits / len(true_positive_jobs))

    return np.mean(precisions), np.mean(recalls), len(precisions)


for k in [1, 3, 5]:
    p, r, n = precision_recall_at_k(k)
    print(f'K={k} | Precision@{k}: {p:.3f} | Recall@{k}: {r:.3f} | (evaluated on {n} students)')

K=1 | Precision@1: 1.000 | Recall@1: 0.853 | (evaluated on 17 students)
K=3 | Precision@3: 0.431 | Recall@3: 1.000 | (evaluated on 17 students)
K=5 | Precision@5: 0.259 | Recall@5: 1.000 | (evaluated on 17 students)


## Cell 26 — Baseline vs Smarter Ranking Comparison

Same Precision@K / Recall@K, but using the baseline (overlap ratio only) ranking,
so we can prove the cosine-similarity ranking is actually better.

In [56]:
def precision_recall_at_k_baseline(k=3):
    precisions, recalls = [], []

    for student_id in students['student_id'].unique():
        true_positive_jobs = set(
            matches[(matches['student_id'] == student_id) & (matches['label'] == 1)]['job_id']
        )
        if not true_positive_jobs:
            continue

        ranked = baseline_rank_jobs_for_student(student_id, top_n=k)
        top_k_jobs = set(ranked['job_id'])

        hits = len(top_k_jobs & true_positive_jobs)
        precisions.append(hits / k)
        recalls.append(hits / len(true_positive_jobs))

    return np.mean(precisions), np.mean(recalls), len(precisions)


print('=== BASELINE vs SMARTER RANKING (Precision@3 / Recall@3) ===')
p_base, r_base, n_base = precision_recall_at_k_baseline(3)
p_smart, r_smart, n_smart = precision_recall_at_k(3)

comparison = pd.DataFrame({
    'Method': ['Baseline (overlap ratio)', 'Smarter (cosine similarity)'],
    'Precision@3': [round(p_base, 3), round(p_smart, 3)],
    'Recall@3': [round(r_base, 3), round(r_smart, 3)]
})
print(comparison.to_string(index=False))

=== BASELINE vs SMARTER RANKING (Precision@3 / Recall@3) ===
                     Method  Precision@3  Recall@3
   Baseline (overlap ratio)        0.431       1.0
Smarter (cosine similarity)        0.431       1.0


## Cell 27 — Plain-English Demo: One Student's Ranked Job List

In [57]:
def explain_ranking_for_student(student_id, top_n=3):
    student = students[students['student_id'] == student_id].iloc[0]
    ranked = rank_jobs_for_student(student_id, top_n=top_n)

    print('=' * 60)
    print(f'RANKED JOBS FOR STUDENT {student_id} (preferred role: {student["preferred_role"]})')
    print('=' * 60)
    for rank, row in ranked.iterrows():
        print(f"Rank {rank}: {row['job_title']} at {row['company_name']} "
              f"(Job {row['job_id']}) — match score {row['match_score']:.3f} "
              f"— thresholds {'PASSED' if row['threshold_passed'] else 'NOT MET'}")
    print()
    top_job_id = ranked.iloc[0]['job_id']
    print(f"Why Rank 1 is top: this job's required skills most closely match "
          f"the student's skill vector in both presence AND score level, "
          f"not just a yes/no overlap.")


explain_ranking_for_student(1, top_n=3)

RANKED JOBS FOR STUDENT 1 (preferred role: Data Analyst)
Rank 1: Data Analyst at TechNova (Job 101) — match score 0.856 — thresholds PASSED
Rank 2: BI Analyst at DataVision (Job 104) — match score 0.523 — thresholds NOT MET
Rank 3: ML Engineer at AI Labs (Job 103) — match score 0.358 — thresholds NOT MET

Why Rank 1 is top: this job's required skills most closely match the student's skill vector in both presence AND score level, not just a yes/no overlap.


## Cell 28 — Plain-English Demo: One Job's Ranked Candidate List

In [58]:
def explain_ranking_for_job(job_id, top_n=3):
    job = jobs[jobs['job_id'] == job_id].iloc[0]
    ranked = rank_students_for_job(job_id, top_n=top_n)

    print('=' * 60)
    print(f'RANKED CANDIDATES FOR JOB {job_id} — {job["job_title"]} at {job["company_name"]}')
    print('=' * 60)
    for rank, row in ranked.iterrows():
        print(f"Rank {rank}: Student {row['student_id']} ({row['preferred_role']}) "
              f"— match score {row['match_score']:.3f} "
              f"— thresholds {'PASSED' if row['threshold_passed'] else 'NOT MET'}")
    print()
    print(f"Why this order: candidates are ranked by how closely their full skill vector "
          f"aligns with this job's required skill vector, not just a single skill match.")


explain_ranking_for_job(101, top_n=3)

RANKED CANDIDATES FOR JOB 101 — Data Analyst at TechNova
Rank 1: Student 6 (Data Analyst) — match score 0.985 — thresholds NOT MET
Rank 2: Student 1 (Data Analyst) — match score 0.856 — thresholds PASSED
Rank 3: Student 19 (Data Analyst) — match score 0.737 — thresholds NOT MET

Why this order: candidates are ranked by how closely their full skill vector aligns with this job's required skill vector, not just a single skill match.


## Cell 29 — Hand-off Summary (Ranking → Frontend search/candidate views)

In [59]:
ranking_handoff = {
    'ranking_method'        : 'cosine similarity on skill vectors (Task 2 vectors reused)',
    'baseline_method'       : 'skill_overlap_ratio threshold',
    'output_for_students'   : 'sorted list of jobs with match_score and threshold_passed flag',
    'output_for_companies'  : 'sorted list of candidates with match_score and threshold_passed flag',
    'metric_used'           : 'Precision@K and Recall@K (K=1,3,5)',
    'precision_at_3_smarter': round(p_smart, 3),
    'recall_at_3_smarter'   : round(r_smart, 3),
    'precision_at_3_baseline': round(p_base, 3),
    'recall_at_3_baseline'  : round(r_base, 3)
}

print('=== RANKING HAND-OFF SUMMARY ===')
for k, v in ranking_handoff.items():
    print(f'{k}: {v}')

=== RANKING HAND-OFF SUMMARY ===
ranking_method: cosine similarity on skill vectors (Task 2 vectors reused)
baseline_method: skill_overlap_ratio threshold
output_for_students: sorted list of jobs with match_score and threshold_passed flag
output_for_companies: sorted list of candidates with match_score and threshold_passed flag
metric_used: Precision@K and Recall@K (K=1,3,5)
precision_at_3_smarter: 0.431
recall_at_3_smarter: 1.0
precision_at_3_baseline: 0.431
recall_at_3_baseline: 1.0
